# Rolling plot of ensemble


In [ ]:
import openmeteo_requests

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import requests_cache
from retry_requests import retry

In [ ]:
import earthkit.data as ekd
import xarray as xr
import dynamical_catalog

In [ ]:
# we are going to use this point (close to Brussels)
latitude = 50.75
longitude = 4.25
location = 'Brussels'

## Fetching observation from the Open Meteo website

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
# code for daily data
arch_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": latitude,  # location of the RMI Uccle weather station (change if you want)
	"longitude": longitude,
    "start_date": "2026-05-01",
	"end_date": "2026-07-27",
	# "hourly": ["temperature_2m"],
    "daily": "temperature_2m_max",
    # "models": "era5",
}
arch_responses = openmeteo.weather_api(arch_url, params = params)
arch_response = arch_responses[0]

In [ ]:
# code for hourly data

# arch_hourly = arch_response.Hourly()
# arch_hourly_temperature_2m = arch_hourly.Variables(0).ValuesAsNumpy()

# arch_hourly_data = {
# 	"date": pd.date_range(
# 		start = pd.to_datetime(arch_hourly.Time(), unit="s", utc=True),
# 		end =  pd.to_datetime(arch_hourly.TimeEnd(), unit="s", utc=True),
# 		freq = pd.Timedelta(arch_hourly.Interval(), unit = "s"),
# 		inclusive = "left"
# 	)
# }

# arch_hourly_data["temperature_2m_obs"] = arch_hourly_temperature_2m

# arch_hourly_dataframe = pd.DataFrame(data = arch_hourly_data).set_index("date")
# observations = arch_hourly_dataframe

In [ ]:
arch_daily = arch_response.Daily()
arch_daily_temperature_2m_max = arch_daily.Variables(0).ValuesAsNumpy()

first_date = pd.to_datetime(arch_daily.Time(), unit="s", utc=True)
end_date = pd.to_datetime(arch_daily.TimeEnd(), unit="s", utc=True)
                            

arch_daily_data = {
	"date": pd.date_range(
		start = first_date,
		end =  end_date,
		freq = pd.Timedelta(arch_daily.Interval(), unit = "s"),
		inclusive = "left"
	)
}

arch_daily_data["temperature_2m_max_obs"] = arch_daily_temperature_2m_max

arch_daily_dataframe = pd.DataFrame(data = arch_daily_data).set_index("date")
observations = arch_daily_dataframe

In [ ]:
# quick plot to check
observations.plot()

## Fetching ensemble forecasts from AWS (yikes) and saving to disk

> **Warning**: this is a maximum temperature build using the hourly temperature, therefore it is not completely accurate, as it ignores what is happening between hours. Therefore it may slightly differs from the actual maximum temperature output of the model. It is still a good approximate estimate of the maximum temperature.

In [ ]:
# fetching the dataset
ds = dynamical_catalog.open("ecmwf-ifs-ens-forecast-15-day-0-25-degree", chunks=None)

In [ ]:
# selecting the summer 2026, then the grid point, and finally resampling to get daily maximum
sel_station = ds.temperature_2m.sel(
    init_time=slice(
        first_date.to_pydatetime().isoformat()[:10],
        end_date.to_pydatetime().isoformat()[:10],
    )
).sel(
    latitude=latitude,
    longitude=longitude,
    method="nearest",
).resample(lead_time='1D').max()

In [ ]:
# reconstructing a time index
valid_time = sel_station['init_time'] + sel_station['lead_time']
sel_station = sel_station.assign_coords(valid_time=(('init_time', 'lead_time'), valid_time.values))

In [ ]:
# saving to zarr format
sel_station.to_zarr(f'IFS_ENS_{location}_heatwaves.zarr')

In [ ]:
# quick plot to test
t=0
plt.figure(figsize=(15,5))
ax = plt.gca()

for i in range(1, len(sel_station.ensemble_member)):
    sel_station.isel(init_time=t, ensemble_member=i).plot(x='valid_time', ax=ax, color='tab:purple', lw=0.8, zorder=-10.)
sel_station.isel(init_time=t, ensemble_member=0).plot(x='valid_time', ax=ax, color='k', lw=2., zorder=0.)
observations.plot(ax=ax, color='tab:green', legend=False, zorder=10., lw=2.)
plt.ylim(0, 45)
plt.xlabel('date')
plt.ylabel('Temperature at 2 metre [°C]')
plt.title('');

## Making a video

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = plt.gca()

def update(t):
    ax.cla()
    arl = list()
    p = observations.plot(ax=ax, color='tab:green', legend=False, zorder=10., lw=2.)
    arl.append(p)

    for i in range(1, len(sel_station.ensemble_member)):
        p = sel_station.isel(init_time=t, ensemble_member=i).plot(x='valid_time', ax=ax, color='tab:purple', zorder=-10., lw=0.8)
        arl.append(p)
    p = sel_station.isel(init_time=t, ensemble_member=0).plot(x='valid_time', ax=ax, color='k', lw=2., zorder=0.)
    arl.append(p)
    plt.title('IFS ENS Forecast initialiazed on the ' + str(sel_station.init_time[t:t+1].to_numpy()[0])[:10] + ' 00 UTC (midnight run)')
    plt.ylim(0, 45)
    plt.xlabel('date')
    plt.ylabel('Temperature at 2 metre [°C]')

    return tuple(arl)

ani = animation.FuncAnimation(fig=fig, func=update, frames=len(sel_station.init_time), interval=200, blit=False)
ani.save('ensemble_evol.mp4')
# ani.to_html5_video()